# Pustka
![image](https://live.staticflickr.com/65535/55223620840_628bd8a082_b.jpg)

## Wstęp

W przestrzeni kosmicznej zwanej Pustką dziesiątki statków zniknęły bez śladu — bez szczątków, bez sygnałów alarmowych. Zespół badawczy "Obserwatorzy Pustki" odkrył, że ostatni raport każdego zaginionego statku zawierał tajemniczą kombinację cech, i wytrenował model AI zdolny ją wykrywać. Zanim sami zaginęli, zdążyli przekazać światu model wraz z jego aktywacjami. Twoim zadaniem jest odkryć, co tak naprawdę wykrywał.

## Zadanie
Raport jest tekstem z wierszami w formacie `concept: value`, na przykład:
```
::GALACTIC REGISTRY ANOMALY REPORT::
ID: epsilon-5
SYSTEM: Rigel
PLANET: Aetheria
SECTOR: Sector 007
PLANETARY_CLASS: Artificial
DOMINANT_FAUNA: Magma-Rock Lizard
DOMINANT_FLORA: Ironwood Tree
NATIVE_SENTIENT: Plantoids
GOVERNMENT: Feudal
TECH_LEVEL: Bio-Tech
PRIMARY_EXPORT: Medical Isotopes
...
```

Każdy wiersz (np. `PLANET: Aetheria`) odpowiada parze `(concept, value)`. Każdy `concept` ma zestaw możliwych wartości `value`. Spośród wszystkich możliwych par `(concept, value)` wybrano 5 ukrytych.

Model językowy został zmodyfikowany, tak aby działał jak klasyfikator binarny, który dla danego raportu zwraca:
- `y = 1`, jeśli **choćby jedna** z  ukrytych 5 par `(concept, value)` pojawiła się w raporcie;
- `y = 0` w przeciwnym razie.

Model ma 100% skuteczność.

Nie masz dostępu do samego modelu. Otrzymujesz natomiast wagi `w` i bias `b` ostatniej (liniowej) warstwy modelu oraz cache aktywacji `activations` (które są wejściem dla ostatniej warstwy) dla każdego raportu ze zbioru walidacyjnego. Predykcję modelu dla raportu `i` możesz wyliczyć jako:
```
logit_i = activations[i] @ w + b
y_i     = 1 if sigmoid(logit_i) >= 0.5 else 0
```

Twoim celem jest zidentyfikowanie wszystkich pięciu par `(concept, value)`, które zostały ukryte.

## Dane

Do dyspozycji masz 3 pliki zbioru walidacyjnego:

**`val_release.jsonl`** — 5 000 raportów, po jednym na wiersz. Każdy rekord zawiera:
- `id` — unikalny identyfikator raportu (integer),
- `sentence` — pełny tekst raportu w formacie `CONCEPT: value`,
- `concepts` — słownik 20 par `concept → value` wyekstrahowanych z tekstu.

Każdy raport ma dokładnie 20 konceptów, każdy z ~10 możliwymi wartościami:


**`val_release_activation_cache.npz`** — cache aktywacji modelu dla tych samych 5 000 raportów, w tej samej kolejności co `val_release.jsonl`. Zawiera:
- `row_ids` — identyfikatory raportów (odpowiadają polu `id` w JSONL),
- `bottleneck_post` — macierz aktywacji, kształtu `(5000, 10)`,
- `out_w`, `out_b` — wagi i bias warstwy wyjściowej.

**`val_release_ground_truth.json`** — lista 5 par `(concept, value)` stanowiących rozwiązanie dla zbioru walidacyjnego. Służy wyłącznie do lokalnej weryfikacji.

Na sprawdzarce nie będzie dostępu do zbioru walidacyjnego. Rozwiązanie zostanie ocenione na ukrytych danych testowych, które mają ten sam format co dane walidacyjne, ale różnią się zawartością (inne raporty, inny model z innymi aktywacjami i wagami oraz inna lista ukrytych par).

## Kryterium Oceny

Oceniamy, ile z 5 ukrytych par `(concept, value)` uda Ci się poprawnie zidentyfikować. Każda para to 20 punktów:

| Poprawnych par | Punkty |
|:--------------:|:------:|
| 5/5 | 100 |
| 4/5 | 80 |
| 3/5 | 60 |
| 2/5 | 40 |
| 1/5 | 20 |
| 0/5 | 0 |

## Ograniczenia

- Dostępne biblioteki: `numpy`, `torch`.
- Ewaluacja na Platformie Konkursowej nie może trwać dłużej niż **1 minuta**.
- Twoje rozwiązanie będzie testowane na Platformie Konkursowej bez dostępu do internetu oraz w środowisku z GPU.

## Pliki Zgłoszeniowe

Należy przesłać wyłącznie **ten notebook uzupełniony o Twoje rozwiązanie** (patrz funkcja `solve_release_set`).

## Ewaluacja

Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`.

Za to zadanie możesz zdobyć pomiędzy 0 a 100 punktów. Liczba punktów, którą zdobędziesz, będzie wyliczona na (tajnym) zbiorze testowym na Platformie Konkursowej na podstawie wyżej wspomnianego wzoru. Jeśli Twoje rozwiązanie nie będzie spełniało powyższych kryteriów lub nie będzie wykonywać się prawidłowo, otrzymasz za zadanie 0 punktów.

## Kod Startowy

In [1]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

FINAL_EVALUATION_MODE = False  # Podczas sprawdzania ustawimy tę flagę na True.

In [2]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

import json
import os
import sys
import numpy as np
import torch
from typing import List, Tuple, Dict, Any

RANDOM_SEED = 2026
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if not FINAL_EVALUATION_MODE:
    DATA_PATH = os.path.join("data", "val_release.jsonl")
    GROUND_TRUTH_PATH = os.path.join("data", "val_release_ground_truth.json")
    ACTIVATION_CACHE_PATH = os.path.join("data", "val_release_activation_cache.npz")
    print(f"Przechowywane aktywacje: {ACTIVATION_CACHE_PATH}")

Przechowywane aktywacje: data/val_release_activation_cache.npz


## Ładowanie Danych

In [3]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    with open(DATA_PATH) as f:
        val_rows = [json.loads(line) for line in f]

    with open(GROUND_TRUTH_PATH) as f:
        GROUND_TRUTH = [tuple(pair) for pair in json.load(f)["trigger_pairs"]]

    print(f"Załadowaliśmy {len(val_rows)} raportów")
    print(f"Załadowaliśmy {len(GROUND_TRUTH)} prawdziwych par wywołujących do lokalnej walidacji")

Załadowaliśmy 5000 raportów
Załadowaliśmy 5 prawdziwych par wywołujących do lokalnej walidacji


In [4]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI #########################

if not FINAL_EVALUATION_MODE:
    if not os.path.isfile(ACTIVATION_CACHE_PATH):
        raise FileNotFoundError(
            f"Brakujący plik z aktywacjami: {ACTIVATION_CACHE_PATH}\n"
            "Umieść go w folderze data/ przed uruchomieniem tego notebooka."
        )

    cache = np.load(ACTIVATION_CACHE_PATH)
    required_keys = {"row_ids", "bottleneck_post", "out_w", "out_b"}
    if set(cache.files) != required_keys:
        raise ValueError(
            f"Cache aktywacji musi zawierać dokładnie {sorted(required_keys)}, otrzymano {sorted(cache.files)}."
        )

    row_ids = cache["row_ids"].astype(int)
    val_activations = cache["bottleneck_post"].astype(np.float32)
    val_w = cache["out_w"].astype(np.float32).reshape(-1)
    val_b = float(cache["out_b"])

    expected_row_ids = np.array([int(row["id"]) for row in val_rows], dtype=int)
    if row_ids.shape != expected_row_ids.shape:
        raise ValueError(
            f"Cache aktywacji zawiera {row_ids.shape[0]} identyfikatorów wierszy, "
            f"ale zbiór danych ma {expected_row_ids.shape[0]} wierszy."
        )
    if not np.array_equal(row_ids, expected_row_ids):
        raise ValueError(
            "Wiersze w cache aktywacji nie są zgodne z val_release.jsonl. "
            "Cache musi być zapisany dokładnie w kolejności zbioru danych."
        )

    print(f"Wczytano cache aktywacji z kluczami: {sorted(cache.files)}")
    print(f"Aktywacje mają wymiar: {val_activations.shape}")
    print(f"Wymiary warstwy wyjściowej: val_w={val_w.shape}, val_b=skalar")

Wczytano cache aktywacji z kluczami: ['bottleneck_post', 'out_b', 'out_w', 'row_ids']
Aktywacje mają wymiar: (5000, 10)
Wymiary warstwy wyjściowej: val_w=(10,), val_b=skalar


In [5]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI #########################

if not FINAL_EVALUATION_MODE:
# Przeanalizujmy jeden przykład
    example = val_rows[0]
    print("\n=== Przykładowy Raport ===")
    print(f"ID: {example.get('id', 'N/A')}")
    print(f"\nText (pierwsze 300 znaków):\n{example['sentence'][:300]}...")
    print("\nConcepts:")
    for concept, value in example["concepts"].items():
        print(f"  {concept}: {value}")
    print(val_activations[0])



=== Przykładowy Raport ===
ID: 0

Text (pierwsze 300 znaków):
::GALACTIC REGISTRY ANOMALY REPORT::
ID: epsilon-5
SYSTEM: Rigel
PLANET: Aetheria
SECTOR: Sector 007
PLANETARY_CLASS: Artificial
DOMINANT_FAUNA: Magma-Rock Lizard
DOMINANT_FLORA: Ironwood Tree
NATIVE_SENTIENT: Plantoids
GOVERNMENT: Feudal
TECH_LEVEL: Bio-Tech
PRIMARY_EXPORT: Medical Isotopes
PRIMARY...

Concepts:
  RegistryID: epsilon-5
  StarSystem: Rigel
  Planet: Aetheria
  Sector: Sector 007
  Classification: Artificial
  PrimaryFauna: Magma-Rock Lizard
  PrimaryFlora: Ironwood Tree
  DominantSentient: Plantoids
  GovernmentType: Feudal
  TechLevel: Bio-Tech
  PrimaryExport: Medical Isotopes
  PrimaryImport: Technology
  ThreatLevel: Cosmic Horror
  StarshipClass: Explorer
  Captain: Adama
  Mission: Espionage
  Cargo: Nothing
  Destination: Capital World
  EncounteredAnomaly: Cosmic String
  LastLogEntry: We are lost.
[2.8166362e-06 3.1963259e-01 4.8223883e-01 2.9666874e-06 2.9038420e-01
 4.4818246e-01 1.5519578e-06 2.

## Kod z Kryterium Oceniającym

In [6]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

def compute_score(
    predicted_pairs: List[Tuple[str, str]],
    ground_truth_pairs: List[Tuple[str, str]],
) -> Dict[str, Any]:
    if len(predicted_pairs) != 5:
        return {
            "score": 0
        }
    pred_set = set(predicted_pairs)
    truth_set = set(ground_truth_pairs)
    correct_pairs = sorted(pred_set & truth_set)
    missed_pairs = sorted(truth_set - pred_set)
    extra_pairs = sorted(pred_set - truth_set)
    return {
        "score": 100.0 * len(correct_pairs) / 5.0,
        "n_correct": len(correct_pairs),
        "n_total": 5,
        "correct_pairs": correct_pairs,
        "missed_pairs": missed_pairs,
        "extra_pairs": extra_pairs,
    }

## Twoje Rozwiązanie
W tej sekcji należy umieścić Twoje rozwiązanie. Wprowadzaj zmiany wyłącznie tutaj!

Nie zmieniaj nazwy tej funkcji ani jej sygnatury (danych wejściowych)


In [7]:
def solve_release_set(
    activations,
    rows,
    w,
    b,
) -> List[Tuple[str, str]]:
    """Zwróć dokładnie 5 krotek (concept, value) ukrytych par."""
    predicted_triggers = [
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
        # ('ConceptName', 'Value'),
    ]
    # TODO: implement me!
    return predicted_triggers

## Ewaluacja
Uruchomienie poniższej komórki pozwoli sprawdzić, ile punktów zdobyłoby Twoje rozwiązanie na danych walidacyjnych. Przed wysłaniem upewnij się, że cały notebook wykonuje się od początku do końca bez błędów i bez konieczności ingerencji użytkownika po wybraniu opcji “Run All”.

Podczas sprawdzania model zostanie oceniony na zbiorze testowym, używając podobnej funkcji ewaluacyjnej.



In [8]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

if not FINAL_EVALUATION_MODE:
    predicted_triggers = solve_release_set(
        activations=val_activations,
        rows=val_rows,
        w=val_w,
        b=val_b,
    )
    print(f"\nTwoje odpowiedzi ({len(predicted_triggers)} par):")
    for i, (concept, value) in enumerate(predicted_triggers, 1):
        print(f"  {i}. {concept} = {value!r}")
    results = compute_score(predicted_triggers, GROUND_TRUTH)
    print(results)
    print(f"Ocena: {results['score']} pkt")


Twoje odpowiedzi (0 par):
{'score': 0}
Ocena: 0 pkt
